# Seaborn Attributes: Examples of Every Attribute
Basics to advanced. Attributes are accessed **without parentheses** (`g.axes`), unlike methods (`g.map()`).

Seaborn sits on top of matplotlib, so most results are matplotlib objects (`Axes`, `Figure`) plus seaborn's own grid objects.

**Sections**
1. Module-level attributes
2. Themes: `axes_style` & `plotting_context`
3. Color palettes
4. Axes-level plots: what they return
5. `FacetGrid` attributes
6. `PairGrid` attributes
7. `JointGrid` attributes
8. `ClusterGrid` attributes
9. Heatmap & colorbar attributes
10. Built-in color collections (`xkcd_rgb`, `crayons`, `cm`)
11. Objects interface (`seaborn.objects`)
12. `rcParams` after `set_theme`

> This notebook builds its own sample data, so it works offline. With internet you can use `sns.load_dataset("tips")` instead.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

print("seaborn    :", sns.__version__)
print("matplotlib :", mpl.__version__)

# Tips-like sample data
rng = np.random.default_rng(42)
n = 120
tips = pd.DataFrame({
    "total_bill": rng.gamma(4, 5, n).round(2),
    "sex":    rng.choice(["Male", "Female"], n),
    "smoker": rng.choice(["Yes", "No"], n),
    "day":    pd.Categorical(rng.choice(["Thur", "Fri", "Sat", "Sun"], n),
                             categories=["Thur", "Fri", "Sat", "Sun"], ordered=True),
    "time":   rng.choice(["Lunch", "Dinner"], n),
    "size":   rng.integers(1, 6, n),
})
tips["tip"] = (tips["total_bill"] * rng.uniform(0.08, 0.25, n)).round(2)
tips.head()

## 1. Module-Level Attributes
| Attribute | Meaning |
|---|---|
| `sns.__version__` | installed version |
| `sns.cm` | seaborn colormaps (`rocket`, `mako`, `flare`, `crest`, `vlag`, `icefire`) |
| `sns.xkcd_rgb` | dict of 954 XKCD color names |
| `sns.crayons` | dict of 120 Crayola colors |
| `sns.palettes.SEABORN_PALETTES` | named qualitative palettes |
| `sns.palettes.QUAL_PALETTE_SIZES` | colors per qualitative palette |
| `sns.objects` | the new objects interface |

In [ ]:
print("version         :", sns.__version__)
print("xkcd_rgb count  :", len(sns.xkcd_rgb), "| 'sky blue' ->", sns.xkcd_rgb["sky blue"])
print("crayons count   :", len(sns.crayons), "| 'Atomic Tangerine' ->", sns.crayons["Atomic Tangerine"])
print("seaborn cmaps   :", [n for n in dir(sns.cm) if not n.startswith("_") and n[0].islower()][:6], "...")
print("SEABORN_PALETTES:", list(sns.palettes.SEABORN_PALETTES))
print("QUAL sizes      :", sns.palettes.QUAL_PALETTE_SIZES)
print("public names    :", len([n for n in dir(sns) if not n.startswith("_")]))

## 2. Themes: `axes_style` & `plotting_context`
`sns.axes_style()` returns the **look** (grid, colors, spines). `sns.plotting_context()` returns the **scale** (font and line sizes).

| Setting | Options |
|---|---|
| style | `darkgrid`, `whitegrid`, `dark`, `white`, `ticks` |
| context | `paper`, `notebook`, `talk`, `poster` |

In [ ]:
style = sns.axes_style("darkgrid")
print("type        :", type(style).__name__)
print("n keys      :", len(style))
for k in ["axes.facecolor", "axes.edgecolor", "axes.grid", "grid.color",
          "axes.spines.top", "xtick.bottom", "font.family"]:
    print(f"{k:18} = {style[k]}")

In [ ]:
# Compare all styles
for s in ["darkgrid", "whitegrid", "dark", "white", "ticks"]:
    st = sns.axes_style(s)
    print(f"{s:9} facecolor={st['axes.facecolor']:10} grid={st['axes.grid']!s:5} "
          f"spines.top={st['axes.spines.top']!s:5} xtick.bottom={st['xtick.bottom']}")

In [ ]:
# Compare all contexts: font and line sizes scale up
for c in ["paper", "notebook", "talk", "poster"]:
    ctx = sns.plotting_context(c)
    print(f"{c:9} font.size={ctx['font.size']:<5} axes.labelsize={ctx['axes.labelsize']:<5} "
          f"lines.linewidth={ctx['lines.linewidth']:<5} figure.figsize? -> n/a")

print("\ncontext keys:", list(sns.plotting_context("notebook"))[:8], "...")

In [ ]:
# Apply temporarily with a `with` block
with sns.axes_style("whitegrid"):
    print("inside  facecolor:", mpl.rcParams["axes.facecolor"])
with sns.plotting_context("talk"):
    print("inside  font.size:", mpl.rcParams["font.size"])

sns.set_theme(style="whitegrid", context="notebook", palette="deep", font_scale=1.0)
print("after set_theme facecolor:", mpl.rcParams["axes.facecolor"])

## 3. Color Palettes
`sns.color_palette()` returns a list-like palette object (`_ColorPalette`).

| Attribute / method | Meaning |
|---|---|
| `len(pal)` | number of colors |
| `pal[i]` | RGB tuple |
| `pal.as_hex()` | list of hex strings |
| `pal.name` / `pal.n_colors` | not defined on all versions; use `len` |
| `pal._repr_html_()` | notebook swatch |

In [ ]:
pal = sns.color_palette("deep")
print("type      :", type(pal).__name__)
print("length    :", len(pal))
print("first RGB :", pal[0])
print("as_hex    :", pal.as_hex())
pal

In [ ]:
# Qualitative palettes: all built-ins
for name in ["deep", "muted", "pastel", "bright", "dark", "colorblind"]:
    p = sns.color_palette(name)
    print(f"{name:10} n={len(p)} first={sns.color_palette(name).as_hex()[0]}")

In [ ]:
# Sequential, diverging, and generated palettes
print("Blues (5)      :", sns.color_palette("Blues", 5).as_hex())
print("viridis (4)    :", sns.color_palette("viridis", 4).as_hex())
print("rocket_r (4)   :", sns.color_palette("rocket_r", 4).as_hex())
print("husl (4)       :", sns.husl_palette(4).as_hex())
print("hls (4)        :", sns.hls_palette(4).as_hex())
print("cubehelix (4)  :", sns.cubehelix_palette(4).as_hex())
print("light 'navy'   :", sns.light_palette("navy", 4).as_hex())
print("dark 'seagreen':", sns.dark_palette("seagreen", 4).as_hex())
print("diverging      :", sns.diverging_palette(220, 20, n=5).as_hex())
print("blend          :", sns.blend_palette(["red", "white", "blue"], 5).as_hex())
print("mpl 'Set2'     :", sns.mpl_palette("Set2", 4).as_hex())
print("as_cmap type   :", type(sns.color_palette("mako", as_cmap=True)).__name__)
print("desat(0.5)     :", sns.desaturate("red", 0.5))
print("set_hls_values :", sns.set_hls_values("red", l=0.7))

In [ ]:
# Global palette & palette context
sns.set_palette("Set2")
print("current cycle:", sns.color_palette().as_hex()[:4], "...")
with sns.color_palette("husl", 3):
    print("inside with  :", sns.color_palette().as_hex())
sns.set_palette("deep")
print("restored     :", sns.color_palette().as_hex()[:3], "...")

## 4. Axes-Level Plots: What They Return
Functions like `scatterplot`, `lineplot`, `barplot`, `boxplot`, `histplot`, `kdeplot` return a matplotlib **`Axes`**. Inspect the result through standard `Axes` attributes.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

ax1 = sns.scatterplot(data=tips, x="total_bill", y="tip", hue="day", ax=axes[0, 0])
ax2 = sns.barplot(data=tips, x="day", y="total_bill", ax=axes[0, 1])
ax3 = sns.boxplot(data=tips, x="day", y="tip", ax=axes[0, 2])
ax4 = sns.histplot(data=tips, x="total_bill", bins=10, kde=True, ax=axes[1, 0])
ax5 = sns.kdeplot(data=tips, x="total_bill", hue="time", ax=axes[1, 1])
ax6 = sns.lineplot(data=tips, x="size", y="tip", ax=axes[1, 2])

print("returns Axes     :", ax1 is axes[0, 0], type(ax1).__name__)
print("scatter  collections:", len(ax1.collections), "->", type(ax1.collections[0]).__name__,
      "| points:", len(ax1.collections[0].get_offsets()) if len(ax1.collections) else 0)
print("scatter  legend labels:", [t.get_text() for t in ax1.get_legend().get_texts()])
print("barplot  patches (bars):", len(ax2.patches), "| lines (error bars):", len(ax2.lines))
print("barplot  bar heights   :", [round(p.get_height(), 1) for p in ax2.patches])
print("boxplot  patches/lines :", len(ax3.patches), "/", len(ax3.lines))
print("histplot patches (bins):", len(ax4.patches), "| lines (kde):", len(ax4.lines))
print("kdeplot  lines         :", len(ax5.lines))
print("lineplot lines/collections:", len(ax6.lines), "/", len(ax6.collections))
print("x tick labels    :", [t.get_text() for t in ax2.get_xticklabels()])
print("axis labels      :", ax2.get_xlabel(), "|", ax2.get_ylabel())

In [ ]:
# Customise through the returned Axes
ax2.set_title("Average bill by day")
sns.despine(ax=ax2)                     # remove top/right spines
print("top spine visible  :", ax2.spines["top"].get_visible())
print("left spine visible :", ax2.spines["left"].get_visible())

# Move the legend outside
sns.move_legend(ax1, "upper left", bbox_to_anchor=(1, 1))
print("legend loc moved   :", ax1.get_legend().get_bbox_to_anchor() is not None)

# Error bars and estimators
ax7 = sns.barplot(data=tips, x="day", y="tip", errorbar="sd")
print("barplot errorbar lines:", len(ax7.lines))
fig2, a = plt.subplots()
sns.countplot(data=tips, x="day", ax=a)
print("countplot counts:", [int(p.get_height()) for p in a.patches])

## 5. `FacetGrid` Attributes
Returned by figure-level functions: `relplot`, `displot`, `catplot`, `lmplot`, and by `sns.FacetGrid(...)`.

| Attribute | Meaning |
|---|---|
| `g.figure` / `g.fig` | matplotlib Figure |
| `g.axes` | 2-D array of Axes |
| `g.axes_dict` | `{facet name: Axes}` |
| `g.ax` | single Axes (only if one facet) |
| `g.axes.flat` | iterate all Axes |
| `g.row_names` / `g.col_names` | facet labels |
| `g.hue_names` | hue levels |
| `g.legend` | legend object |
| `g.data` | source DataFrame |
| `g.col_wrap` | wrap setting |
| `g.row_var` / `g.col_var` / `g.hue_var` | variable names |

In [ ]:
g = sns.relplot(data=tips, x="total_bill", y="tip", col="time", row="smoker", hue="sex",
                kind="scatter", height=3, aspect=1.2)

print("type           :", type(g).__name__)
fig_attr = g.figure if hasattr(g, "figure") else g.fig
print("figure         :", fig_attr)
print("axes shape     :", g.axes.shape)
print("axes_dict keys :", list(g.axes_dict))
print("row_names      :", g.row_names)
print("col_names      :", g.col_names)
print("hue_names      :", g.hue_names)
print("row/col/hue var:", g.row_var if hasattr(g, "row_var") else g._row_var,
      g.col_var if hasattr(g, "col_var") else g._col_var,
      g.hue_var if hasattr(g, "hue_var") else g._hue_var)
print("legend         :", g.legend)
print("data shape     :", g.data.shape)
print("number of axes :", len(g.axes.flat))
print("first axes title:", g.axes[0, 0].get_title())

In [ ]:
# Single-facet grid exposes .ax
g1 = sns.displot(data=tips, x="total_bill", kde=True)
print("g1.ax            :", g1.ax)
print("g1.axes.shape    :", g1.axes.shape)

# Wrapped columns
g2 = sns.catplot(data=tips, x="sex", y="tip", col="day", col_wrap=2, kind="bar", height=2.5)
print("col_wrap         :", g2.axes.shape, "| col_names:", g2.col_names)
print("axes_dict        :", list(g2.axes_dict))

# Edit every facet
g2.set_titles("{col_name}")
g2.set_axis_labels("Sex", "Mean tip")
for ax in g2.axes.flat:
    ax.axhline(2, color="grey", ls="--")
print("title now        :", g2.axes.flat[0].get_title())

## 6. `PairGrid` Attributes
Returned by `sns.pairplot()` and `sns.PairGrid()`.

| Attribute | Meaning |
|---|---|
| `g.figure` / `g.fig` | Figure |
| `g.axes` | 2-D array of Axes |
| `g.diag_axes` | diagonal Axes (or `None`) |
| `g.x_vars` / `g.y_vars` | variables for columns / rows |
| `g.diag_vars` | variables on the diagonal |
| `g.hue_names` | hue levels |
| `g.data` | source DataFrame |
| `g.square_grid` | x_vars == y_vars? |

In [ ]:
g = sns.pairplot(tips[["total_bill", "tip", "size", "time"]], hue="time", corner=False)

print("type        :", type(g).__name__)
print("axes shape  :", g.axes.shape)
print("x_vars      :", list(g.x_vars))
print("y_vars      :", list(g.y_vars))
print("diag_vars   :", g.diag_vars)
print("diag_axes   :", None if g.diag_axes is None else len(g.diag_axes), "axes")
print("hue_names   :", g.hue_names)
print("square_grid :", g.square_grid)
print("data cols   :", list(g.data.columns))
print("axes[0,1] x/y labels:", g.axes[0, 1].get_xlabel(), "|", g.axes[0, 1].get_ylabel())

In [ ]:
# Rectangular PairGrid (different x and y variables)
g = sns.PairGrid(tips, x_vars=["total_bill", "size"], y_vars=["tip"], hue="smoker")
g.map(sns.scatterplot)
g.add_legend()
print("axes shape :", g.axes.shape)
print("square_grid:", g.square_grid)
print("legend     :", g.legend is not None or getattr(g, "_legend", None) is not None)

## 7. `JointGrid` Attributes
Returned by `sns.jointplot()` and `sns.JointGrid()`.

| Attribute | Meaning |
|---|---|
| `g.ax_joint` | central Axes |
| `g.ax_marg_x` | top marginal Axes |
| `g.ax_marg_y` | right marginal Axes |
| `g.figure` / `g.fig` | Figure |
| `g.x` / `g.y` | the plotted data |
| `g.hue` | hue data |

In [ ]:
g = sns.jointplot(data=tips, x="total_bill", y="tip", kind="scatter", hue="time")
print("type       :", type(g).__name__)
print("ax_joint   :", g.ax_joint)
print("ax_marg_x  :", g.ax_marg_x)
print("ax_marg_y  :", g.ax_marg_y)
print("x[:3]      :", list(g.x[:3]))
print("y[:3]      :", list(g.y[:3]))
print("hue present:", g.hue is not None)
print("axes in fig:", len((g.figure if hasattr(g, 'figure') else g.fig).axes))
print("joint xlabel/ylabel:", g.ax_joint.get_xlabel(), "|", g.ax_joint.get_ylabel())

In [ ]:
# Build a custom JointGrid
g = sns.JointGrid(data=tips, x="total_bill", y="tip", height=5, ratio=3, space=0.1)
g.plot_joint(sns.kdeplot, fill=True)
g.plot_marginals(sns.histplot, kde=True)
g.set_axis_labels("Bill ($)", "Tip ($)")
print("ratio / space applied; joint ylabel:", g.ax_joint.get_ylabel())

## 8. `ClusterGrid` Attributes
Returned by `sns.clustermap()` (needs SciPy).

| Attribute | Meaning |
|---|---|
| `g.ax_heatmap` | heatmap Axes |
| `g.ax_row_dendrogram` / `g.ax_col_dendrogram` | dendrogram Axes |
| `g.ax_cbar` | colorbar Axes |
| `g.dendrogram_row` / `g.dendrogram_col` | dendrogram objects |
| `g.data2d` | reordered data matrix |
| `g.data` | original data |
| `g.mask` | mask used |

In [ ]:
corr = tips[["total_bill", "tip", "size"]].assign(
    extra=rng.normal(size=n)).corr()

try:
    g = sns.clustermap(corr, annot=True, cmap="vlag", center=0, figsize=(5, 5))
    print("type            :", type(g).__name__)
    print("ax_heatmap      :", g.ax_heatmap)
    print("ax_cbar         :", g.ax_cbar)
    print("row dendrogram  :", type(g.dendrogram_row).__name__)
    print("row order       :", g.dendrogram_row.reordered_ind)
    print("col order       :", g.dendrogram_col.reordered_ind)
    print("linkage shape   :", g.dendrogram_row.linkage.shape)
    print("data2d (reordered):\n", g.data2d.round(2))
    print("x tick labels   :", [t.get_text() for t in g.ax_heatmap.get_xticklabels()])
except ImportError as e:
    print("clustermap needs scipy:", e)

## 9. Heatmap & Colorbar Attributes
`sns.heatmap()` returns an `Axes`. The colored grid is a `QuadMesh` in `ax.collections[0]`.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
hm = sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                 linewidths=0.5, cbar_kws={"label": "correlation"}, ax=ax)

mesh = hm.collections[0]
print("returns Axes    :", hm is ax)
print("mesh type       :", type(mesh).__name__)
print("mesh array shape:", mesh.get_array().shape)
print("cmap name       :", mesh.get_cmap().name)
print("clim            :", mesh.get_clim())
print("norm vmin/vmax  :", mesh.norm.vmin, mesh.norm.vmax)
print("annotations     :", len(hm.texts), "text labels")
print("first annotation:", hm.texts[0].get_text())
print("x labels        :", [t.get_text() for t in hm.get_xticklabels()])
print("y labels        :", [t.get_text() for t in hm.get_yticklabels()])
print("axes in figure  :", len(fig.axes), "(heatmap + colorbar)")
cbar_ax = fig.axes[-1]
print("colorbar ylabel :", cbar_ax.get_ylabel())
print("colorbar attr   :", mesh.colorbar.orientation if mesh.colorbar else None)

print("\nmasked upper triangle example:")
mask = np.triu(np.ones_like(corr, dtype=bool))
fig2, ax2 = plt.subplots()
sns.heatmap(corr, mask=mask, ax=ax2)
print("masked cells    :", int(np.ma.getmaskarray(ax2.collections[0].get_array()).sum()))

## 10. Built-in Color Collections
`sns.xkcd_rgb`, `sns.crayons` (dicts), `sns.cm` (colormaps), and `sns.xkcd_palette()` / `sns.crayon_palette()`.

In [ ]:
print("xkcd_palette  :", sns.xkcd_palette(["denim blue", "pale red", "medium green"]).as_hex())
print("crayon_palette:", sns.crayon_palette(["Atomic Tangerine", "Cerulean"]).as_hex())

for name in ["rocket", "mako", "flare", "crest", "vlag", "icefire"]:
    cm = getattr(sns.cm, name)
    print(f"{name:8} type={type(cm).__name__:20} N={cm.N} name={cm.name}")

print("\nxkcd sample:", dict(list(sns.xkcd_rgb.items())[:4]))
print("crayons sample:", dict(list(sns.crayons.items())[:4]))

## 11. Objects Interface (`seaborn.objects`)
Newer, composable API (seaborn 0.12+). A `Plot` is built with `.add()` and rendered with `.plot()`.

In [ ]:
try:
    import seaborn.objects as so
    import dataclasses

    p = (so.Plot(tips, x="total_bill", y="tip", color="time")
           .add(so.Dot(alpha=0.7))
           .add(so.Line(color=".2"), so.PolyFit(order=1), color=None)
           .facet(col="smoker")
           .label(x="Bill", y="Tip", title="Tips")
           .scale(color="Set2")
           .layout(size=(8, 4)))

    print("Plot type          :", type(p).__name__)
    print("Mark fields (Dot)  :", [f.name for f in dataclasses.fields(so.Dot)][:8])
    print("Mark fields (Line) :", [f.name for f in dataclasses.fields(so.Line)][:8])
    print("Move/Stat examples :", so.Dodge().__class__.__name__, so.PolyFit().__class__.__name__,
          so.Agg().__class__.__name__, so.Jitter().__class__.__name__)

    plotter = p.plot()                       # renders
    fig_obj = plotter._figure
    print("Rendered figure axes:", len(fig_obj.axes))
    print("Axes titles         :", [a.get_title() for a in fig_obj.axes])
except ImportError as e:
    print("seaborn.objects needs seaborn >= 0.12:", e)

## 12. `rcParams` After `set_theme`
`sns.set_theme()` writes into `matplotlib.rcParams`. `sns.reset_defaults()` and `sns.reset_orig()` undo it.

In [ ]:
sns.set_theme(style="darkgrid", context="talk", font="sans-serif", font_scale=1.2,
              rc={"axes.titleweight": "bold"})
for k in ["axes.facecolor", "axes.grid", "font.size", "axes.labelsize",
          "axes.titleweight", "lines.linewidth", "figure.figsize"]:
    print(f"{k:18} = {mpl.rcParams[k]}")

sns.reset_orig()                   # back to matplotlib defaults
print("\nafter reset_orig facecolor:", mpl.rcParams["axes.facecolor"])

sns.reset_defaults()               # seaborn's own defaults
print("after reset_defaults font.size:", mpl.rcParams["font.size"])

# sns.set() is the old name of set_theme()
sns.set_theme()
print("current style facecolor:", sns.axes_style()["axes.facecolor"])

## Summary
- **Module:** `__version__`, `cm`, `xkcd_rgb`, `crayons`, `palettes.SEABORN_PALETTES`
- **Themes:** `axes_style()`, `plotting_context()`, `set_theme()`, `reset_orig()`, `reset_defaults()`
- **Palettes:** `color_palette()`, `.as_hex()`, `husl_palette`, `cubehelix_palette`, `light_palette`, `diverging_palette`
- **Axes-level plots return `Axes`:** `ax.patches`, `ax.lines`, `ax.collections`, `ax.get_legend()`
- **`FacetGrid`:** `figure`, `axes`, `axes_dict`, `ax`, `row_names`, `col_names`, `hue_names`, `legend`, `data`
- **`PairGrid`:** `axes`, `diag_axes`, `x_vars`, `y_vars`, `diag_vars`, `square_grid`
- **`JointGrid`:** `ax_joint`, `ax_marg_x`, `ax_marg_y`, `x`, `y`
- **`ClusterGrid`:** `ax_heatmap`, `ax_cbar`, `dendrogram_row`, `dendrogram_col`, `data2d`
- **Heatmap:** `ax.collections[0]` (QuadMesh), `get_array()`, `get_clim()`
- **Objects API:** `so.Plot`, marks, `.add()`, `.facet()`, `.scale()`, `.plot()`